In [ ]:
import pandas as pd
import numpy as np
import requests

def fetch_lbins(item_ids):
    def get_price(item):
        data = requests.get(
            f"https://sky.coflnet.com/api/auctions/tag/{item}/active/bin"
        ).json()
        return data[0]["startingBid"] if data else 0

    return {item: get_price(item) for item in item_ids}

bazaar = {
    k: {
        "Sell": v["quick_status"]["sellPrice"],
        "Buy": v["quick_status"]["buyPrice"]
    }
    for k, v in requests.get("https://api.hypixel.net/v2/skyblock/bazaar").json()['products'].items()
}

def apply_price_overrides(df,price_map,weight_col="Weight",price_col="Price"):
    df[price_col] *= 1.35

    mask = df["Item"].isin(price_map)
    df.loc[mask, price_col] = df.loc[mask, "Item"].map(price_map)

    weights = df[weight_col]
    factor = weights / weights.sum()

    df[price_col] *= factor

lbin_items = {
    'SNOW_SUIT_HELMET','SNOW_SUIT_CHESTPLATE','SNOW_SUIT_LEGGINGS','SNOW_SUIT_BOOTS','GIFT_THE_FISH','GOLD_GIFT','NEW_BOTTLE_OF_JYRRE','PET_SNOWMAN','CRYOPOWDER_SHARD','WINTER_ISLAND'
}
lbin_prices = fetch_lbins(lbin_items)

snow_minion_cost = bazaar['SNOW_BLOCK']['Sell']*992 + bazaar['ENCHANTED_SNOW_BLOCK']['Sell']*248
lbin_prices["Snow Minion"] = 250000-snow_minion_cost #I was able to sell bulk t11 snow minions for 250k

white = pd.read_csv("white.csv")
green = pd.read_csv("green.csv")
red = pd.read_csv("red.csv")

apply_price_overrides(white, lbin_prices)
apply_price_overrides(green, lbin_prices)
apply_price_overrides(red, lbin_prices)

print(f"Expected Profit For White: {(2 * white["Price"].sum() - bazaar["WHITE_GIFT"]['Sell']):,.0f}")
print(f"Expected Profit For Green: {(2 * green["Price"].sum() - bazaar["GREEN_GIFT"]['Sell']):,.0f}")
print(f"Expected Profit For Red: {(2 * red["Price"].sum() - bazaar["RED_GIFT"]['Sell']):,.0f}")

Red: {'Buy': 4291, 'Sell': 7473}, max cost: 34177.31954633116
Green: {'Buy': 11082, 'Sell': 12208}, max cost: 19941.707717583387
White: {'Buy': 3865, 'Sell': 4066}, max cost: 7876.388935033403
